In [ ]:
# =============================================================================
# jobs/22_pipeline_health.py
# Is the pipeline working? One row per check, rebuilt on every run.
#
# A failing notebook tells you something broke. It does not tell you that
# nothing ran, which is the quieter failure: a paused schedule, a cluster that
# never started, an upstream feed that stopped. These checks measure the age
# and completeness of what the pipeline produces, so silence shows up too.
#
# Checks (thresholds are in config.py, HEALTH_*):
#   fgs_latest_statement        age of the newest FGS statement
#   fgs_intersections_pending   statements without up-to-date intersections
#   situation_table_age         age of constituency_flood_situation (proves the
#                               polling chain, 07 to 19, ran recently)
#   warnings_current_age        age of the newest flood warnings ingest
#   ea_live_readings_age        age of the newest live EA reading (23). WARNS at most,
#                               never fails: an EA API outage must not fail the run
#   warnings_unlinked           current warnings that match no constituency
#   mp_details_age / mp_count   freshness and completeness of MP details
#   constituency_count          completeness of the boundary table
#   mps_unmatched               constituencies with no matched MP (vacancies are
#                               normal; a jump is not)
#   exposure_tables             the exposure tables (jobs 33 to 35) exist, are not older than
#                               HEALTH_EXPOSURE_WARN_DAYS, and cover every warning and alert
#                               area. INFO when not built yet, WARN at most, never fails
#   exposure_people_total       the people the exposure work counts across England are near the
#                               Census total (a broken join shows here). WARN at most
#
# status: ok, info (worth knowing, not a problem), warn, fail.
# The table is written first. The notebook then raises if any check is "fail", so
# a scheduled run is marked failed and your alerts fire.
#
# Exit values: "success" (no check failed).
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType
from config import (
    TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_CONST_SITUATION, TBL_FLOOD_WARN_CURRENT,
    TBL_WARNING_CONTACTS, TBL_MP_CONTACTS, TBL_EA_FWA, TBL_EA_FAA, TBL_AREA_EXPOSURE, TBL_AREA_COVERAGE, TBL_AREA_GEOGRAPHIES, TBL_AREA_SITES, TBL_EXPOSURE_GRID, TBL_CONSTITUENCIES, TBL_PIPELINE_HEALTH, TBL_EA_RT_LIVE, TBL_RFG_UPDATES, TBL_RFG_EXPOSURE,
    FGS_GEOMETRY_SOURCE, MIN_CONSTITUENCIES, MIN_MPS,
    HEALTH_FGS_WARN_HOURS, HEALTH_FGS_FAIL_HOURS,
    HEALTH_WARNINGS_WARN_HOURS, HEALTH_WARNINGS_FAIL_HOURS,
    HEALTH_CONTACTS_WARN_HOURS, HEALTH_CONTACTS_FAIL_HOURS,
    HEALTH_MP_WARN_DAYS, HEALTH_MP_FAIL_DAYS, HEALTH_LIVE_WARN_HOURS,
    HEALTH_EXPOSURE_WARN_DAYS, HEALTH_EXPOSURE_PEOPLE_MIN, HEALTH_EXPOSURE_PEOPLE_MAX,
)
from utils.helpers import get_spark, utc_now, source_predicate, table_exists


spark = get_spark()
now   = utc_now()

ORDER   = {"ok": 0, "info": 1, "warn": 2, "fail": 3}
results = []   # (check_name, status, value, detail)


def parse_ts(value):
    """A datetime or ISO string (with or without Z) as an aware UTC datetime."""
    if isinstance(value, datetime):
        dt = value
    else:
        dt = datetime.fromisoformat(str(value).replace("Z", "+00:00"))
    return dt if dt.tzinfo else dt.replace(tzinfo=timezone.utc)


def by_age(age, warn, fail):
    return "fail" if age > fail else "warn" if age > warn else "ok"


def check(name, fn):
    """Run one check. A check that cannot run is itself a failure."""
    try:
        status, value, detail = fn()
    except Exception as e:
        status, value, detail = "fail", "", f"could not run: {str(e)[:300]}"
    results.append((name, status, str(value), detail))
    print(f"  [{status:>4}] {name}: {value}  {detail}")


In [ ]:
# =============================================================================
# THE CHECKS
# =============================================================================

print(f"Pipeline health at {now.isoformat()}")


def fgs_latest_statement():
    newest = spark.sql(f"SELECT MAX(issued_at) AS m FROM {TBL_STATEMENTS}").collect()[0]["m"]
    if newest is None:
        return "fail", "", "no statements held"
    age = (now - parse_ts(newest)).total_seconds() / 3600
    return (by_age(age, HEALTH_FGS_WARN_HOURS, HEALTH_FGS_FAIL_HOURS), f"{age:.1f} h",
            f"newest statement issued {newest}; warn over {HEALTH_FGS_WARN_HOURS} h, fail over {HEALTH_FGS_FAIL_HOURS} h")


def fgs_intersections_pending():
    in_source = source_predicate(spark, TBL_FGS_INTERSECT_LOG, FGS_GEOMETRY_SOURCE, alias="l")
    n = spark.sql(f"""
        SELECT COUNT(*) AS n
        FROM {TBL_STATEMENTS} s
        LEFT JOIN (SELECT l.statement_id, l.last_modified_at
                   FROM {TBL_FGS_INTERSECT_LOG} l WHERE {in_source}) l
          ON l.statement_id = s.statement_id
         AND COALESCE(l.last_modified_at, '') = COALESCE(s.last_modified_at, '')
        WHERE l.statement_id IS NULL
    """).collect()[0]["n"]
    status = "ok" if n == 0 else "fail" if n >= 10 else "warn"
    return status, n, f"statements for geometry source '{FGS_GEOMETRY_SOURCE}' awaiting 07_compute_intersections"


def situation_table_age():
    newest = spark.sql(f"SELECT MAX(computed_at) AS m FROM {TBL_CONST_SITUATION}").collect()[0]["m"]
    if newest is None:
        return "fail", "", "constituency_flood_situation is empty"
    age = (now - parse_ts(newest)).total_seconds() / 3600
    return (by_age(age, HEALTH_CONTACTS_WARN_HOURS, HEALTH_CONTACTS_FAIL_HOURS), f"{age:.1f} h",
            f"last built {newest}; warn over {HEALTH_CONTACTS_WARN_HOURS} h, fail over {HEALTH_CONTACTS_FAIL_HOURS} h")


def warnings_current_age():
    newest = spark.sql(f"SELECT MAX(ingested_at) AS m FROM {TBL_FLOOD_WARN_CURRENT}").collect()[0]["m"]
    if newest is None:
        return "info", "", "no warnings or alerts in force, so the age of the ingest cannot be read from this table"
    age = (now - parse_ts(newest)).total_seconds() / 3600
    return (by_age(age, HEALTH_WARNINGS_WARN_HOURS, HEALTH_WARNINGS_FAIL_HOURS), f"{age:.1f} h",
            f"newest ingest {newest}; warn over {HEALTH_WARNINGS_WARN_HOURS} h, fail over {HEALTH_WARNINGS_FAIL_HOURS} h")


def ea_live_readings_age():
    # Warn at most. A stale feed is usually the EA API, and a "fail" here would fail the
    # whole polling run, which carries the FGS and warnings alerts.
    if not table_exists(spark, TBL_EA_RT_LIVE):
        return "info", "", "ea_rt_readings_live does not exist yet (23_ea_rt_live_readings has not run)"
    newest = spark.sql(f"SELECT MAX(ingested_at) AS m FROM {TBL_EA_RT_LIVE}").collect()[0]["m"]
    if newest is None:
        return "info", "", "no live readings held yet"
    age = (now - parse_ts(newest)).total_seconds() / 3600
    return ("ok" if age <= HEALTH_LIVE_WARN_HOURS else "warn"), f"{age:.1f} h", \
           f"newest live reading ingested {newest}; warn over {HEALTH_LIVE_WARN_HOURS} h (never fails)"


def rfg_uncounted():
    # Information only. RFG is a side feature and is quiet for long spells; a count of updates job 37 has not yet counted must not raise the app-wide banner.
    if not table_exists(spark, TBL_RFG_UPDATES):
        return "info", "", "rfg_updates does not exist yet (36_ingest_rfg has not run)"
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {TBL_RFG_UPDATES} WHERE is_present AND n_polygons > 0").collect()[0]["n"]
    if not table_exists(spark, TBL_RFG_EXPOSURE):
        return ("ok" if n == 0 else "info"), n, "RFG updates with polygons; none counted yet (37_rfg_exposure has not run)"
    m = spark.sql(f"SELECT COUNT(*) AS n FROM {TBL_RFG_UPDATES} u LEFT JOIN (SELECT DISTINCT update_key FROM {TBL_RFG_EXPOSURE}) e ON e.update_key = u.update_key WHERE u.is_present AND u.n_polygons > 0 AND e.update_key IS NULL").collect()[0]["n"]
    return ("ok" if m == 0 else "info"), m, "RFG updates with polygons that job 37 has not counted yet (information only: a side feature never raises the app banner)"


def warnings_unlinked():
    n = spark.sql(f"""
        SELECT COUNT(DISTINCT w.flood_area_id) AS n
        FROM {TBL_FLOOD_WARN_CURRENT} w
        LEFT JOIN (SELECT DISTINCT flood_area_id FROM {TBL_WARNING_CONTACTS}) c
          ON c.flood_area_id = w.flood_area_id
        WHERE c.flood_area_id IS NULL
    """).collect()[0]["n"]
    return ("ok" if n == 0 else "warn"), n, "current warnings with no constituency row in flood_warning_constituency_contacts"


def mp_details_age():
    newest = spark.sql(f"SELECT MAX(last_refreshed_at) AS m FROM {TBL_MP_CONTACTS}").collect()[0]["m"]
    if newest is None:
        return "fail", "", "mp_contact_details is empty"
    age = (now - parse_ts(newest)).total_seconds() / 86400
    return (by_age(age, HEALTH_MP_WARN_DAYS, HEALTH_MP_FAIL_DAYS), f"{age:.1f} days",
            f"last refreshed {newest}; warn over {HEALTH_MP_WARN_DAYS} days, fail over {HEALTH_MP_FAIL_DAYS} days")


def mp_count():
    n = spark.table(TBL_MP_CONTACTS).count()
    return ("ok" if n >= MIN_MPS else "fail"), n, f"expected at least {MIN_MPS}"


def constituency_count():
    n = spark.table(TBL_CONSTITUENCIES).count()
    return ("ok" if n >= MIN_CONSTITUENCIES else "fail"), n, f"expected at least {MIN_CONSTITUENCIES}"


def mps_unmatched():
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {TBL_CONST_SITUATION} WHERE NOT mp_matched").collect()[0]["n"]
    return ("ok" if n == 0 else "info" if n <= 10 else "warn"), n, "constituencies with no matched MP (vacancies are normal; a jump is not)"


EXPOSURE_TABLES = (TBL_AREA_EXPOSURE, TBL_AREA_COVERAGE, TBL_AREA_GEOGRAPHIES, TBL_AREA_SITES, TBL_EXPOSURE_GRID)


def exposure_tables():
    # The exposure products are built by hand and read by the app's Warning areas pages. WARN at most: they must never fail the polling run.
    missing = [t.split(".")[-1] for t in EXPOSURE_TABLES if not table_exists(spark, t)]
    if len(missing) == len(EXPOSURE_TABLES):
        return "info", "not built", "none of the exposure tables exist yet (jobs 33 to 35 have not run)"
    problems = []
    if missing:
        problems.append("missing: " + ", ".join(missing))
    oldest = None
    for t in EXPOSURE_TABLES:
        if table_exists(spark, t):
            m = spark.sql(f"DESCRIBE DETAIL {t}").collect()[0]["lastModified"]
            oldest = m if oldest is None or parse_ts(m) < parse_ts(oldest) else oldest
    age_days = (now - parse_ts(oldest)).total_seconds() / 86400
    if age_days > HEALTH_EXPOSURE_WARN_DAYS:
        problems.append(f"oldest table last built {age_days:.0f} days ago (warn over {HEALTH_EXPOSURE_WARN_DAYS})")
    n_areas = spark.table(TBL_EA_FWA).count() + spark.table(TBL_EA_FAA).count()
    n_rows = spark.table(TBL_AREA_EXPOSURE).count() if table_exists(spark, TBL_AREA_EXPOSURE) else 0
    if n_rows < 0.9 * n_areas:
        problems.append(f"ea_area_exposure has {n_rows:,} rows for {n_areas:,} areas")
    return ("warn" if problems else "ok"), f"{age_days:.0f} days", "; ".join(problems) or f"{n_rows:,} areas; oldest table {age_days:.0f} days old"


def exposure_people_total():
    if not table_exists(spark, TBL_AREA_COVERAGE):
        return "info", "", "ea_area_coverage does not exist yet"
    total = spark.sql(f"SELECT people FROM {TBL_AREA_COVERAGE} WHERE kind = 'all'").collect()
    if not total:
        return "warn", "", "ea_area_coverage has no row for all of England"
    n = float(total[0]["people"])
    ok = HEALTH_EXPOSURE_PEOPLE_MIN <= n <= HEALTH_EXPOSURE_PEOPLE_MAX
    return ("ok" if ok else "warn"), f"{n:,.0f}", f"people counted across England; expected {HEALTH_EXPOSURE_PEOPLE_MIN:,} to {HEALTH_EXPOSURE_PEOPLE_MAX:,}"


for name, fn in (
    ("fgs_latest_statement",      fgs_latest_statement),
    ("fgs_intersections_pending", fgs_intersections_pending),
    ("situation_table_age",       situation_table_age),
    ("warnings_current_age",      warnings_current_age),
    ("ea_live_readings_age",      ea_live_readings_age),
    ("rfg_uncounted",             rfg_uncounted),
    ("warnings_unlinked",         warnings_unlinked),
    ("mp_details_age",            mp_details_age),
    ("mp_count",                  mp_count),
    ("constituency_count",        constituency_count),
    ("mps_unmatched",             mps_unmatched),
    ("exposure_tables",           exposure_tables),
    ("exposure_people_total",     exposure_people_total),
):
    check(name, fn)

worst = max((r[1] for r in results), key=lambda s: ORDER[s])
failed = [r[0] for r in results if r[1] == "fail"]
results.append(("overall", worst, "", "worst of the checks above" + (f"; failing: {', '.join(failed)}" if failed else "")))
print(f"\nOverall: {worst}")


In [ ]:
# =============================================================================
# WRITE, THEN RAISE IF ANYTHING FAILED
# =============================================================================

schema = StructType([
    StructField("check_name",  StringType(), True),
    StructField("status",      StringType(), True),
    StructField("value",       StringType(), True),
    StructField("detail",      StringType(), True),
    StructField("checked_at",  StringType(), True),
])

(
    spark.createDataFrame([(n, s, v, d, now.isoformat()) for n, s, v, d in results], schema)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TBL_PIPELINE_HEALTH)
)

spark.sql(f"""
    COMMENT ON TABLE {TBL_PIPELINE_HEALTH} IS
    'One row per pipeline health check, rebuilt on every run by 22_pipeline_health.
     status: ok, info, warn, fail. The overall row is the worst of the rest. Checks
     measure the age and completeness of what the pipeline produces, so a pipeline
     that has stopped running shows up as well as one that is failing.'
""")

if failed:
    raise Exception("Pipeline health: failing checks: " + ", ".join(failed))

dbutils.notebook.exit("success")
